In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib, pandas as pd

INTER = pathlib.Path("/content/drive/MyDrive/precos-combustiveis-anp/intermediarios")
df = pd.read_parquet(INTER / "unido_bruto.parquet")
funil = [("linhas lidas (7 arquivos)", len(df))]

df = df.drop(columns=["nome_da_rua", "valor_compra"])
print(df.shape)

Mounted at /content/drive
(3038685, 11)


In [2]:
vazias = df["data_coleta"].isna() & df["valor_venda"].isna()
print("linhas vazias:", vazias.sum())
df = df[~vazias]
funil.append(("sem linhas vazias", len(df)))

linhas vazias: 9114


In [3]:
raw = df["data_coleta"]
d1 = pd.to_datetime(raw, format="%d/%m/%Y", errors="coerce")
d2 = pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
df["data_coleta"] = d1.fillna(d2)
print("datas inválidas:", df["data_coleta"].isna().sum())
print(df["data_coleta"].min(), "->", df["data_coleta"].max())

datas inválidas: 0
2023-01-02 00:00:00 -> 2026-06-30 00:00:00


In [4]:
antes = df["valor_venda"].notna().sum()
df["valor_venda"] = pd.to_numeric(
    df["valor_venda"].str.replace(",", ".", regex=False), errors="coerce")
print("viraram nulos na conversão:", antes - df["valor_venda"].notna().sum())
df = df[df["valor_venda"].notna()]
funil.append(("com preço de venda válido", len(df)))

viraram nulos na conversão: 0


In [5]:
df = df[df["unidade"].str.contains("litro", case=False, na=False)]
funil.append(("somente R$/litro (sem GNV)", len(df)))

MAPA_PRODUTO = {
    "GASOLINA": "gasolina",
    "GASOLINA ADITIVADA": "gasolina_aditivada",
    "ETANOL": "etanol",
    "DIESEL S10": "diesel_s10",
    "DIESEL": "diesel",
}
df["produto"] = df["produto"].map(MAPA_PRODUTO)
print(df["produto"].value_counts(dropna=False))

produto
gasolina              788453
etanol                666801
gasolina_aditivada    613318
diesel_s10            556388
diesel                340500
Name: count, dtype: int64


In [6]:
chave = ["cnpj", "produto", "data_coleta"]
print("mesmo posto+produto+dia:", df.duplicated(subset=chave).sum())
print("idênticos também no preço:", df.duplicated(subset=chave + ["valor_venda"]).sum())
df[df.duplicated(subset=chave, keep=False)].sort_values(chave).head(10)

mesmo posto+produto+dia: 18
idênticos também no preço: 14


,regiao,uf,municipio,revenda,cnpj,produto,data_coleta,valor_venda,unidade,bandeira,arquivo_origem
1328001,SE,SP,TAUBATE,AUTO POSTO MANGUEIRA II LTDA,00.008.226/0001-39,gasolina,2024-06-10,5.48,R$ / litro,VIBRA,ca-2024-01.zip
1328003,SE,SP,TAUBATE,AUTO POSTO MANGUEIRA II LTDA,00.008.226/0001-39,gasolina,2024-06-10,5.65,R$ / litro,VIBRA,ca-2024-01.zip
1338624,SE,SP,TAUBATE,AUTO POSTO MANGUEIRA II LTDA,00.008.226/0001-39,gasolina,2024-06-12,5.48,R$ / litro,VIBRA,ca-2024-01.zip
1338626,SE,SP,TAUBATE,AUTO POSTO MANGUEIRA II LTDA,00.008.226/0001-39,gasolina,2024-06-12,5.65,R$ / litro,VIBRA,ca-2024-01.zip
1328002,SE,SP,TAUBATE,AUTO POSTO MANGUEIRA II LTDA,00.008.226/0001-39,gasolina_aditivada,2024-06-10,5.59,R$ / litro,VIBRA,ca-2024-01.zip
1328004,SE,SP,TAUBATE,AUTO POSTO MANGUEIRA II LTDA,00.008.226/0001-39,gasolina_aditivada,2024-06-10,5.77,R$ / litro,VIBRA,ca-2024-01.zip
1338625,SE,SP,TAUBATE,AUTO POSTO MANGUEIRA II LTDA,00.008.226/0001-39,gasolina_aditivada,2024-06-12,5.59,R$ / litro,VIBRA,ca-2024-01.zip
1338627,SE,SP,TAUBATE,AUTO POSTO MANGUEIRA II LTDA,00.008.226/0001-39,gasolina_aditivada,2024-06-12,5.77,R$ / litro,VIBRA,ca-2024-01.zip
1328425,SE,RJ,BARRA MANSA,POSTO METANO LTDA,03.328.588/0001-03,gasolina,2024-06-10,5.59,R$ / litro,VIBRA,ca-2024-01.zip
1328427,SE,RJ,BARRA MANSA,POSTO METANO LTDA,03.328.588/0001-03,gasolina,2024-06-10,5.59,R$ / litro,VIBRA,ca-2024-01.zip


In [7]:
print(df.groupby("produto")["valor_venda"]
        .describe(percentiles=[.001, .01, .5, .99, .999]).round(2))

                       count  mean   std   min  0.1%    1%   50%   99%  99.9%  \
produto                                                                         
diesel              340500.0  6.06  0.59  3.97  4.49  4.69  5.99  7.79   8.44   
diesel_s10          556388.0  6.14  0.61  4.19  4.49  4.74  6.06  7.99   8.75   
etanol              666801.0  4.24  0.57  2.63  2.88  3.08  4.19  5.69   6.08   
gasolina            788453.0  5.99  0.55  4.09  4.54  4.79  5.99  7.40   8.49   
gasolina_aditivada  613318.0  6.18  0.56  3.47  4.68  4.97  6.19  7.59   8.97   

                     max  
produto                   
diesel              9.45  
diesel_s10          9.99  
etanol              7.44  
gasolina            9.79  
gasolina_aditivada  9.89  


In [8]:
n0 = len(df)
df = df.drop_duplicates(subset=["cnpj", "produto", "data_coleta"], keep="first")
print("removidas:", n0 - len(df))
funil.append(("sem duplicados posto+produto+dia", len(df)))

removidas: 18


In [9]:
cnpj_limpo = df["cnpj"].str.replace(r"\D", "", regex=True)
print("CNPJ com tamanho diferente de 14:", (cnpj_limpo.str.len() != 14).sum())
print("UF nula:", df["uf"].isna().sum(), "| município nulo:", df["municipio"].isna().sum())
print("UFs distintas:", df["uf"].nunique())

CNPJ com tamanho diferente de 14: 0
UF nula: 0 | município nulo: 0
UFs distintas: 27


In [10]:
df["ano"] = df["data_coleta"].dt.year
df["mes"] = df["data_coleta"].dt.month
df["ano_mes"] = df["data_coleta"].dt.to_period("M").astype(str)

In [11]:
med = df.groupby(["produto", "uf", "ano_mes"])["valor_venda"].transform("median")
razao = df["valor_venda"] / med
suspeito = (razao < 0.7) | (razao > 1.3)

print("suspeitos:", suspeito.sum(), f"({suspeito.mean()*100:.3f}%)")
print(df[suspeito].groupby("produto").size())

(df[suspeito].assign(mediana=med[suspeito].round(2))
   .sort_values("valor_venda")
   [["uf", "municipio", "revenda", "produto", "data_coleta", "valor_venda", "mediana"]]
   .head(15))

suspeitos: 9303 (0.314%)
produto
diesel                 306
diesel_s10            1335
etanol                3234
gasolina              2242
gasolina_aditivada    2186
dtype: int64


,uf,municipio,revenda,produto,data_coleta,valor_venda,mediana
2889757,SP,LINS,AUTO POSTO DO TUCA COMERCIO DE COMBUSTIVEIS LTDA,etanol,2026-04-29,3.06,4.48
2891329,SP,LINS,AUTO POSTO TAKAO LINS LTDA,etanol,2026-04-29,3.09,4.48
2892691,SP,LINS,AUTO POSTO CASTELO DE LINS LTDA,etanol,2026-04-29,3.09,4.48
2886815,SP,LINS,AUTO POSTO CINQUENTENARIO DE LINS LTDA,etanol,2026-04-29,3.09,4.48
2107637,CE,SOBRAL,POSTO PRINCESA DO NORTE LTDA,etanol,2025-05-13,3.39,5.35
68041,SP,PRESIDENTE PRUDENTE,AUTO POSTO RODOTRUCK DE PRESIDENTE PRUDENTE LTDA,gasolina_aditivada,2023-01-30,3.47,5.14
33534,SP,SOROCABA,AUTO POSTO JARDIM IBITI LTDA,gasolina_aditivada,2023-01-17,3.49,5.14
64036,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,gasolina_aditivada,2023-01-31,3.53,5.14
599157,RJ,SANTO ANTONIO DE PADUA,AUTO POSTO BOA NOVA DE PÁDUA LTDA,diesel,2023-09-04,4.09,5.89
617152,RJ,SANTO ANTONIO DE PADUA,AUTO POSTO BOA NOVA DE PÁDUA LTDA,diesel,2023-09-12,4.09,5.89


In [12]:
razao = df["valor_venda"] / med
bins = [0, .5, .6, .7, 1.3, 1.5, 2, 100]
print(pd.cut(razao, bins=bins).value_counts().sort_index())

cols = ["uf", "municipio", "revenda", "produto", "data_coleta", "valor_venda"]
ext = df.assign(mediana=med.round(2), razao=razao.round(2)).sort_values("razao")
print("--- 8 MAIS BAIXOS em relação à mediana ---")
print(ext.head(8)[cols + ["mediana", "razao"]])
print("--- 8 MAIS ALTOS em relação à mediana ---")
print(ext.tail(8)[cols + ["mediana", "razao"]])

valor_venda
(0.0, 0.5]            0
(0.5, 0.6]            0
(0.6, 0.7]           15
(0.7, 1.3]      2956139
(1.3, 1.5]         8234
(1.5, 2.0]         1054
(2.0, 100.0]          0
Name: count, dtype: int64
--- 8 MAIS BAIXOS em relação à mediana ---
         uf               municipio  \
2107637  CE                  SOBRAL   
689081   RJ  SANTO ANTONIO DE PADUA   
707180   RJ  SANTO ANTONIO DE PADUA   
743164   RJ  SANTO ANTONIO DE PADUA   
68041    SP     PRESIDENTE PRUDENTE   
2889757  SP                    LINS   
33534    SP                SOROCABA   
671174   RJ  SANTO ANTONIO DE PADUA   

                                                  revenda             produto  \
2107637                      POSTO PRINCESA DO NORTE LTDA              etanol   
689081                  AUTO POSTO BOA NOVA DE PÁDUA LTDA              diesel   
707180                  AUTO POSTO BOA NOVA DE PÁDUA LTDA              diesel   
743164                  AUTO POSTO BOA NOVA DE PÁDUA LTDA              dies

In [13]:
med = df.groupby(["produto", "uf", "ano_mes"])["valor_venda"].transform("median")
razao = df["valor_venda"] / med

# 1) remover erros prováveis
erro_provavel = razao > 1.5
print("removidas (preço > 1,5x a mediana):", erro_provavel.sum())
df = df[~erro_provavel].copy()
funil.append(("sem preços > 1,5x a mediana", len(df)))

# 2) marcar atípicos que ficam na base
razao = razao[~erro_provavel]
df["preco_atipico"] = (razao < 0.7) | (razao > 1.3)
print("marcadas como atípicas:", df["preco_atipico"].sum())

# 3) funil e salvamento
print(pd.DataFrame(funil, columns=["etapa", "linhas"]).to_string(index=False))

SAIDA = pathlib.Path("/content/drive/MyDrive/precos-combustiveis-anp/dados_tratados")
SAIDA.mkdir(exist_ok=True)
df = df.reset_index(drop=True)
df.to_parquet(SAIDA / "precos_tratados.parquet", index=False)
print(df.shape)
print(df.dtypes)

removidas (preço > 1,5x a mediana): 1054
marcadas como atípicas: 8249
                           etapa  linhas
       linhas lidas (7 arquivos) 3038685
               sem linhas vazias 3029571
       com preço de venda válido 3029571
      somente R$/litro (sem GNV) 2965460
sem duplicados posto+produto+dia 2965442
     sem preços > 1,5x a mediana 2964388
(2964388, 15)
regiao                    object
uf                        object
municipio                 object
revenda                   object
cnpj                      object
produto                   object
data_coleta       datetime64[ns]
valor_venda              float64
unidade                   object
bandeira                  object
arquivo_origem            object
ano                        int32
mes                        int32
ano_mes                   object
preco_atipico               bool
dtype: object
